# Khojki release verification
CPU-only. Upload `khojki-quality-input.zip` in Files, then run cells in order. Save and verify the result archive before disconnecting. This notebook changes no historical experiment or frozen model.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
BUNDLE = Path('/content/khojki-quality-input.zip')
assert BUNDLE.is_file(), 'Upload the prepared ZIP through Files before running this cell.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == '4c0be8da5c46d7261dc8a1571b67a09c5e4687279fead5ed3f630ca66e0c5b2e'
ROOT = Path('/content/khojki-quality')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    for name in z.namelist():
        assert not Path(name).is_absolute() and '..' not in Path(name).parts
    z.extractall(ROOT)
manifest = json.loads((ROOT/'quality-input-manifest.json').read_text())
for name, expected in manifest.items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == expected, name
for key in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS']:
    os.environ[key] = '2'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT)+'[test,train]'], check=True)
print('Verified input files:', len(manifest), 'Bundle SHA256:', '4c0be8da5c46d7261dc8a1571b67a09c5e4687279fead5ed3f630ca66e0c5b2e')


In [ ]:
output = ROOT/'results/quality-validation'
log = ROOT/'quality-run.log'
with log.open('w') as stream:
    completed = subprocess.run([sys.executable, str(ROOT/'scripts/verify_quality.py'), '--private-data', '--output', str(output)], stdout=stream, stderr=subprocess.STDOUT)
print(log.read_text()[-5000:])
print('Return code:', completed.returncode)
if (output/'verification.json').exists():
    print((output/'verification.json').read_text())


In [ ]:
import base64
from IPython.display import HTML, display
archive = Path('/content/khojki-quality-results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(log, 'quality-run.log')
    z.write(ROOT/'quality-input-manifest.json', 'quality-input-manifest.json')
    for file in output.rglob('*'):
        if file.is_file() and not file.is_symlink() and not any(parent.is_symlink() for parent in file.parents if parent != output):
            z.write(file, str(file.relative_to(ROOT)))
raw = archive.read_bytes()
print(json.dumps({'name': archive.name, 'bytes':len(raw), 'sha256':hashlib.sha256(raw).hexdigest()}))
display(HTML('<a download="'+archive.name+'" href="data:application/zip;base64,'+base64.b64encode(raw).decode()+'">Save verified quality results</a>'))
